# Part 4: Training & Experiments

**Goal:** Run controlled experiments to understand what makes self-attention work.

---

## 4.1 Setup

We'll use the boosted parameters discovered in Part 3 to ensure real gradient flow.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# Core functions (self-contained — no external dependencies)
# ============================================================

def generate_argmax_sample(dim=4, value_range=(0, 10), seed=None):
    if seed is not None:
        np.random.seed(seed)
    x = np.random.randint(value_range[0], value_range[1], size=(dim,))
    return x, np.full((dim,), np.max(x))


def row_softmax(scores: np.ndarray) -> np.ndarray:
    scores = scores - np.max(scores, axis=1, keepdims=True)
    exp_scores = np.exp(scores)
    return exp_scores / np.sum(exp_scores, axis=1, keepdims=True)


def attention_forward(hidden_layer, W_Q, W_K, W_V, W_H, return_cache=False, softmax_fn=None):
    if softmax_fn is None:
        softmax_fn = row_softmax
    d_k = W_Q.shape[1]
    Q, K, V = hidden_layer @ W_Q, hidden_layer @ W_K, hidden_layer @ W_V
    scores = (Q @ K.T) / np.sqrt(d_k)
    A = softmax_fn(scores)
    Z = A @ V
    H = Z @ W_H
    if not return_cache:
        return H
    return H, {"Q": Q, "K": K, "V": V, "scores": scores, "A": A, "Z": Z, "H": H}


def mse(pred, target):
    return np.mean((pred - target) ** 2)


def attention_step(W_Q, W_K, W_V, W_H, hidden_layer, target, lr=1e-3, clip=1.0, softmax_fn=None):
    pred, cache = attention_forward(hidden_layer, W_Q, W_K, W_V, W_H, return_cache=True, softmax_fn=softmax_fn)
    loss = mse(pred, target)
    dH = (2.0 / pred.size) * (pred - target)
    
    Z, A, V, Q, K = cache["Z"], cache["A"], cache["V"], cache["Q"], cache["K"]
    dW_H = Z.T @ dH
    dZ = dH @ W_H.T
    dV = A.T @ dZ
    dA = dZ @ V.T
    dS = A * (dA - np.sum(dA * A, axis=1, keepdims=True))
    
    d_k = W_Q.shape[1]
    dQ = (dS @ K) / np.sqrt(d_k)
    dK = (dS.T @ Q) / np.sqrt(d_k)
    dW_Q, dW_K, dW_V = hidden_layer.T @ dQ, hidden_layer.T @ dK, hidden_layer.T @ dV
    
    grad_norms = {"W_Q": float(np.linalg.norm(dW_Q)), "W_K": float(np.linalg.norm(dW_K)),
                  "W_V": float(np.linalg.norm(dW_V)), "W_H": float(np.linalg.norm(dW_H))}
    
    dW_Q, dW_K = np.clip(dW_Q, -clip, clip), np.clip(dW_K, -clip, clip)
    dW_V, dW_H = np.clip(dW_V, -clip, clip), np.clip(dW_H, -clip, clip)
    
    return W_Q - lr*dW_Q, W_K - lr*dW_K, W_V - lr*dW_V, W_H - lr*dW_H, loss, cache, grad_norms


def make_hidden_and_target(x_raw, y_raw, W_embed, scale=10.0):
    x = (np.array(x_raw, dtype=np.float64).reshape(-1, 1)) / scale
    y = (np.array(y_raw, dtype=np.float64).reshape(-1, 1)) / scale
    return x @ W_embed, y @ W_embed


def decode_embedding_to_scalar(embedded, W_embed, scale=10.0):
    w = W_embed.reshape(-1)
    return (embedded @ w) / (w @ w + 1e-12) * scale


def attention_row_entropy(A: np.ndarray) -> float:
    eps = 1e-12
    P = np.clip(A, eps, 1.0)
    return float(np.mean(-np.sum(P * np.log(P), axis=1)))


def attention_row_entropy_normalized(A: np.ndarray) -> float:
    """Normalized entropy in [0,1]: 0=sharp, 1=uniform."""
    eps = 1e-12
    P = np.clip(A, eps, 1.0)
    P = P / np.sum(P, axis=1, keepdims=True)
    H = -np.sum(P * np.log(P), axis=1)
    return float(np.mean(H / (np.log(P.shape[1]) + eps)))


print("All functions loaded.")

In [ ]:
# Global configuration
d_model = 8
init_scale = 0.1

# Boosted parameters (from Part 3)
DEFAULT_SCALE = 1.0   # NOT 10.0
DEFAULT_LR = 1e-2     # NOT 1e-3
DEFAULT_CLIP = 10.0   # NOT 1.0

# Dataset
N = 400
min_len, max_len = 2, 8

rng_data = np.random.default_rng(123)
dataset = []
for _ in range(N):
    L = int(rng_data.integers(min_len, max_len + 1))
    x, y = generate_argmax_sample(dim=L, value_range=(0, 10))
    dataset.append((x, y))

train_frac = 0.8
train_N = int(N * train_frac)
train_set = dataset[:train_N]
test_set = dataset[train_N:]

print(f"Dataset: {N} samples, train={len(train_set)}, test={len(test_set)}")
print(f"Sequence lengths: {min_len}–{max_len}")

# Embedding (shared across experiments)
rng = np.random.default_rng(0)
W_embed = init_scale * rng.standard_normal((1, d_model))

## 4.2 Helper Functions for Experiments

In [ ]:
def attention_forward_alpha(hidden_layer, W_Q, W_K, W_V, W_H, alpha=1.0, return_cache=False):
    """
    Forward pass with temperature scaling.
    
    alpha > 1 → sharper attention (lower entropy)
    alpha < 1 → smoother attention (higher entropy)
    """
    d_k = W_Q.shape[1]
    Q = hidden_layer @ W_Q
    K = hidden_layer @ W_K
    V = hidden_layer @ W_V
    scores = alpha * (Q @ K.T) / np.sqrt(d_k)  # alpha scales BEFORE softmax
    A = row_softmax(scores)
    Z = A @ V
    H = Z @ W_H
    if not return_cache:
        return H
    cache = {"Q": Q, "K": K, "V": V, "scores": scores, "A": A, "Z": Z, "H": H}
    return H, cache


def attention_step_split_lr(W_Q, W_K, W_V, W_H, hidden_layer, target,
                            alpha=1.0, lr_qk=1e-3, lr_vh=1e-3, clip=1.0):
    """
    SGD step with separate learning rates for Q/K vs V/H.
    
    Hypothesis: Q/K gradients can be "starved" relative to V/H.
    Giving Q/K a higher lr may help attention structure emerge.
    """
    pred, cache = attention_forward_alpha(hidden_layer, W_Q, W_K, W_V, W_H, 
                                          alpha=alpha, return_cache=True)
    loss = mse(pred, target)
    dH = (2.0 / pred.size) * (pred - target)
    
    Z, A, V, Q, K = cache["Z"], cache["A"], cache["V"], cache["Q"], cache["K"]
    dW_H = Z.T @ dH
    dZ = dH @ W_H.T
    
    dV = A.T @ dZ
    dA = dZ @ V.T
    dS = A * (dA - np.sum(dA * A, axis=1, keepdims=True))
    
    d_k = W_Q.shape[1]
    dQ = (dS @ K) / np.sqrt(d_k)
    dK = (dS.T @ Q) / np.sqrt(d_k)
    
    dW_Q = hidden_layer.T @ dQ
    dW_K = hidden_layer.T @ dK
    dW_V = hidden_layer.T @ dV
    
    grad_norms = {
        "W_Q": float(np.linalg.norm(dW_Q)),
        "W_K": float(np.linalg.norm(dW_K)),
        "W_V": float(np.linalg.norm(dW_V)),
        "W_H": float(np.linalg.norm(dW_H)),
    }
    
    dW_Q = np.clip(dW_Q, -clip, clip)
    dW_K = np.clip(dW_K, -clip, clip)
    dW_V = np.clip(dW_V, -clip, clip)
    dW_H = np.clip(dW_H, -clip, clip)
    
    # Different learning rates!
    W_Q = W_Q - lr_qk * dW_Q
    W_K = W_K - lr_qk * dW_K
    W_V = W_V - lr_vh * dW_V
    W_H = W_H - lr_vh * dW_H
    
    return W_Q, W_K, W_V, W_H, float(loss), cache, grad_norms

In [ ]:
def train_and_evaluate(alpha=1.0, epochs=300, lr=None, lr_qk=None, lr_vh=None,
                       clip=DEFAULT_CLIP, scale=DEFAULT_SCALE, seed=0):
    """
    Full training run with all metrics tracked.
    
    Returns dict with: loss_hist, entropy_hist, grad_norm_hist, 
                       final weights, test metrics
    """
    # Handle learning rate defaults
    if lr is not None:
        lr_qk = lr_qk or lr
        lr_vh = lr_vh or lr
    else:
        lr_qk = lr_qk or DEFAULT_LR
        lr_vh = lr_vh or DEFAULT_LR
    
    # Initialize weights
    rng = np.random.default_rng(seed)
    wq = init_scale * rng.standard_normal((d_model, d_model))
    wk = init_scale * rng.standard_normal((d_model, d_model))
    wv = init_scale * rng.standard_normal((d_model, d_model))
    wh = init_scale * rng.standard_normal((d_model, d_model))
    
    loss_hist = []
    entropy_hist = []
    grad_norm_hist = {"W_Q": [], "W_K": [], "W_V": [], "W_H": []}
    
    for _ in range(epochs):
        ep_losses, ep_entropies = [], []
        ep_gnorms = {k: [] for k in grad_norm_hist}
        
        for x_raw, y_raw in train_set:
            hidden, target = make_hidden_and_target(x_raw, y_raw, W_embed, scale=scale)
            wq, wk, wv, wh, loss, cache, gnorms = attention_step_split_lr(
                wq, wk, wv, wh, hidden, target,
                alpha=alpha, lr_qk=lr_qk, lr_vh=lr_vh, clip=clip
            )
            if np.isfinite(loss):
                ep_losses.append(loss)
                ep_entropies.append(attention_row_entropy_normalized(cache["A"]))
                for k in ep_gnorms:
                    ep_gnorms[k].append(gnorms[k])
        
        loss_hist.append(float(np.mean(ep_losses)) if ep_losses else np.nan)
        entropy_hist.append(float(np.mean(ep_entropies)) if ep_entropies else np.nan)
        for k in grad_norm_hist:
            grad_norm_hist[k].append(float(np.mean(ep_gnorms[k])) if ep_gnorms[k] else np.nan)
    
    # Test evaluation
    test_losses, test_entropies = [], []
    for x_raw, y_raw in test_set[:50]:
        hidden, target = make_hidden_and_target(x_raw, y_raw, W_embed, scale=scale)
        pred, cache = attention_forward_alpha(hidden, wq, wk, wv, wh, alpha=alpha, return_cache=True)
        test_losses.append(mse(pred, target))
        test_entropies.append(attention_row_entropy_normalized(cache["A"]))
    
    return {
        "alpha": float(alpha),
        "epochs": int(epochs),
        "lr_qk": float(lr_qk),
        "lr_vh": float(lr_vh),
        "final_train_loss": float(loss_hist[-1]),
        "test_mse": float(np.mean(test_losses)),
        "test_entropy": float(np.mean(test_entropies)),
        "loss_hist": loss_hist,
        "entropy_hist": entropy_hist,
        "grad_norm_hist": grad_norm_hist,
        "weights": (wq, wk, wv, wh),
    }


def make_label(r):
    """Create readable label for plot legends."""
    if r["lr_qk"] != r["lr_vh"]:
        return f"α={r['alpha']}, qk={r['lr_qk']:.0e}, vh={r['lr_vh']:.0e}"
    return f"α={r['alpha']}, lr={r['lr_qk']:.0e}"

In [ ]:
def plot_results(results):
    """Plot loss, entropy, and gradient norms for multiple experiments."""
    styles = ["-o", "--s", "-.^", ":D", "-v", "--x"]
    
    # Loss + entropy
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for i, r in enumerate(results):
        label = make_label(r)
        ms = styles[i % len(styles)]
        axes[0].plot(r["loss_hist"], ms, markevery=20, markersize=5, label=label)
        axes[1].plot(r["entropy_hist"], ms, markevery=20, markersize=5, label=label)
    
    axes[0].set_yscale("log")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Train loss")
    axes[0].set_title("Loss curves")
    axes[0].legend()
    axes[0].grid(True)
    
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Normalized entropy")
    axes[1].set_title("Attention entropy (0=sharp, 1=uniform)")
    axes[1].legend()
    axes[1].grid(True)
    
    plt.tight_layout()
    plt.show()
    
    # Gradient norms
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    for idx, wname in enumerate(["W_Q", "W_K", "W_V", "W_H"]):
        ax = axes[idx // 2][idx % 2]
        for i, r in enumerate(results):
            label = make_label(r)
            ms = styles[i % len(styles)]
            ax.plot(r["grad_norm_hist"][wname], ms, markevery=20, markersize=5, label=label)
        ax.set_yscale("log")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Grad norm")
        ax.set_title(f"Gradient norms: {wname}")
        ax.legend(fontsize=8)
        ax.grid(True)
    
    plt.tight_layout()
    plt.show()

---

## 4.3 Experiment A: Temperature Scaling ($\alpha$)

### Hypothesis

Scaling attention scores by $\alpha$ before softmax changes the sharpness:

$$
A = \text{softmax}\left(\alpha \cdot \frac{QK^T}{\sqrt{d_k}}\right)
$$

- $\alpha > 1$ → larger score differences → **sharper** attention (lower entropy)
- $\alpha < 1$ → smaller score differences → **smoother** attention (higher entropy)

**Prediction:** Higher $\alpha$ should help the model focus on the argmax position more quickly.

In [ ]:
# Experiment A configurations
configs_A = [
    {"alpha": 0.5, "epochs": 200, "lr": DEFAULT_LR},
    {"alpha": 1.0, "epochs": 200, "lr": DEFAULT_LR},
    {"alpha": 2.0, "epochs": 200, "lr": DEFAULT_LR},
    {"alpha": 4.0, "epochs": 200, "lr": DEFAULT_LR},
]

print("Running Experiment A: Temperature Scaling")
print("=" * 50)

results_A = []
for cfg in configs_A:
    r = train_and_evaluate(**cfg, seed=0)
    results_A.append(r)
    print(f"α={r['alpha']:.1f}: train_loss={r['final_train_loss']:.6f}, "
          f"test_mse={r['test_mse']:.6f}, entropy={r['test_entropy']:.4f}")

In [ ]:
plot_results(results_A)

In [ ]:
# Summary plot: alpha vs metrics
alphas = [r["alpha"] for r in results_A]
test_mse = [r["test_mse"] for r in results_A]
test_ent = [r["test_entropy"] for r in results_A]

fig, ax = plt.subplots(1, 2, figsize=(10, 3))

ax[0].plot(alphas, test_mse, "o-", linewidth=2, markersize=8)
ax[0].set_xlabel("α (temperature)")
ax[0].set_ylabel("Test MSE")
ax[0].set_title("Test Error vs Temperature")
ax[0].grid(True)

ax[1].plot(alphas, test_ent, "o-", linewidth=2, markersize=8, color="tab:orange")
ax[1].set_xlabel("α (temperature)")
ax[1].set_ylabel("Test Entropy")
ax[1].set_title("Attention Sharpness vs Temperature")
ax[1].grid(True)

plt.tight_layout()
plt.show()

### Experiment A Interpretation

*Fill in after running:*

- **Observation:** ...
- **Best $\alpha$:** ...
- **Why?** ...

---

## 4.4 Experiment B: Split Learning Rates

### Hypothesis

In attention, gradients flow:
- **Loss → W_H → W_V → W_K/W_Q** (long path for Q/K)

The Q/K matrices control *which positions attend to which*. The V/H matrices control *what values are passed*.

**Gradient starvation:** Q/K may receive smaller gradients because they're further from the loss.

**Fix:** Give Q/K a higher learning rate to compensate.

In [ ]:
# Experiment B configurations
configs_B = [
    # Baseline: uniform lr
    {"alpha": 2.0, "epochs": 200, "lr_qk": 1e-2, "lr_vh": 1e-2},
    # Boost Q/K
    {"alpha": 2.0, "epochs": 200, "lr_qk": 2e-2, "lr_vh": 1e-2},
    {"alpha": 2.0, "epochs": 200, "lr_qk": 5e-2, "lr_vh": 1e-2},
    # Boost V/H instead (control)
    {"alpha": 2.0, "epochs": 200, "lr_qk": 1e-2, "lr_vh": 5e-2},
]

print("Running Experiment B: Split Learning Rates")
print("=" * 50)

results_B = []
for cfg in configs_B:
    r = train_and_evaluate(**cfg, seed=0)
    results_B.append(r)
    print(f"lr_qk={r['lr_qk']:.0e}, lr_vh={r['lr_vh']:.0e}: "
          f"train_loss={r['final_train_loss']:.6f}, test_mse={r['test_mse']:.6f}")

In [ ]:
plot_results(results_B)

### Experiment B Interpretation

*Fill in after running:*

- **Does boosting lr_qk help?** ...
- **Does boosting lr_vh help?** ...
- **Conclusion about gradient starvation:** ...

---

## 4.5 Inspecting Trained Models

Let's examine attention patterns from the best model.

In [ ]:
# Pick best result from Experiment A
best_A = min(results_A, key=lambda r: r["test_mse"])
wq, wk, wv, wh = best_A["weights"]

print(f"Best model from Exp A: α={best_A['alpha']}, test_mse={best_A['test_mse']:.6f}")
print("\n" + "="*60)

# Show predictions on test examples
for i in range(min(5, len(test_set))):
    x_raw, y_raw = test_set[i]
    hidden, target = make_hidden_and_target(x_raw, y_raw, W_embed, scale=DEFAULT_SCALE)
    pred, cache = attention_forward_alpha(hidden, wq, wk, wv, wh, 
                                          alpha=best_A["alpha"], return_cache=True)
    
    yhat = decode_embedding_to_scalar(pred, W_embed, scale=DEFAULT_SCALE)
    
    print(f"\nExample {i+1}:")
    print(f"  Input:      {x_raw}")
    print(f"  Target:     {y_raw} (max={np.max(x_raw)})")
    print(f"  Prediction: {np.round(yhat, 2)}")
    print(f"  Attention row sums: {np.round(cache['A'].sum(axis=1), 4)}")
    
    # Does prediction match the max?
    error = np.abs(yhat - y_raw).mean()
    if error < 1.0:
        print(f"  ✓ Good prediction (error={error:.2f})")
    else:
        print(f"  ✗ Poor prediction (error={error:.2f})")

In [ ]:
# Visualize attention matrix
x_raw, y_raw = test_set[0]
hidden, target = make_hidden_and_target(x_raw, y_raw, W_embed, scale=DEFAULT_SCALE)
pred, cache = attention_forward_alpha(hidden, wq, wk, wv, wh,
                                      alpha=best_A["alpha"], return_cache=True)

A = cache["A"]
n = A.shape[0]

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(A, cmap="Blues", vmin=0, vmax=1)
ax.set_xlabel("Key position")
ax.set_ylabel("Query position")
ax.set_title(f"Attention Matrix\nx={x_raw}, max={np.max(x_raw)}")
plt.colorbar(im, ax=ax, label="Attention weight")

# Mark argmax position
argmax_pos = np.argmax(x_raw)
ax.axvline(argmax_pos, color="red", linestyle="--", linewidth=2, label=f"argmax={argmax_pos}")
ax.legend()

plt.tight_layout()
plt.show()

print(f"\nIdeally, all rows should concentrate weight on column {argmax_pos} (the argmax position).")

---

## 4.6 Summary & Report Template

### What We Learned

1. **Broadcasting bug:** Missing `keepdims=True` breaks softmax normalization

2. **Gradient signal:** Input scaling (`scale`) dramatically affects gradient magnitude; tiny gradients = no learning

3. **Temperature ($\alpha$):** Higher values → sharper attention → faster convergence on this task

4. **Split learning rates:** May help if Q/K gradients are starved relative to V/H

### Report Structure

```
1. Background
   - Self-attention mechanism
   - The argmax broadcast task

2. Failure Analysis
   - Broadcasting bug: what broke and why
   - Gradient signal issue: why entropy was constant

3. Fixes Applied
   - keepdims=True in softmax
   - Boosted parameters (scale, lr, clip)

4. Experiments
   - Experiment A: Temperature scaling
     - Hypothesis → Method → Results → Interpretation
   - Experiment B: Split learning rates
     - Hypothesis → Method → Results → Interpretation

5. Results
   - Best configuration found
   - Attention patterns inspection
   - Test set performance

6. Limitations
   - Single-head attention only
   - Toy task (may not generalize)
   - No systematic hyperparameter search
```

### Key Hyperparameters (Final Reference)

| Parameter | Default | Effect | Tuning Advice |
|-----------|---------|--------|---------------|
| `scale` | 10.0 → **1.0** | Input normalization | Lower if gradients vanish |
| `lr` | 1e-3 → **1e-2** | Learning rate | Raise if loss plateaus |
| `clip` | 1.0 → **10.0** | Gradient clipping | Raise if gradients are clipped |
| `alpha` | 1.0 | Temperature | Higher → sharper attention |
| `lr_qk` | = lr | Q/K learning rate | Raise if attention won't focus |
| `lr_vh` | = lr | V/H learning rate | Usually keep equal to lr |